# Mini RAG walkthrough
Small LLM (Qwen2.5-0.5B) + MiniLM embeddings + FAISS.

**Pipeline:** load → chunk → embed → index → retrieve → generate

In [ ]:
# !pip install -r requirements.txt   # uncomment in Colab

## 1. Load & chunk documents

In [ ]:
from rag_core import load_chunks

chunks = load_chunks('data')
print(len(chunks), 'chunks')
print(chunks[0])

## 2. Load models and build the FAISS index
First run downloads the models (~1.2 GB).

In [ ]:
from rag_core import MiniRAG

rag = MiniRAG()
print('Indexed:', rag.build_index(chunks))

## 3. Retrieval only
See which chunks match a query.

In [ ]:
for chunk, score in rag.retrieve('What is FAISS?', k=3):
    print(f'{score:.3f} | {chunk.source} | {chunk.text[:120]}...')

## 4. Full RAG answer

In [ ]:
answer, hits = rag.answer('Why is RAG useful with small language models?')
print(answer)
print('\nSources:', [c.source for c, _ in hits])

## 5. Compare with and without retrieval
The small model alone is more likely to hallucinate.

In [ ]:
import torch
q = 'What index type does FAISS use for exact cosine search?'

# Without context
msgs = [{'role': 'user', 'content': q}]
prompt = rag.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inp = rag.tokenizer(prompt, return_tensors='pt')
with torch.inference_mode():
    out = rag.llm.generate(**inp, max_new_tokens=100, do_sample=False)
print('NO RAG :', rag.tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True))
print('WITH RAG:', rag.answer(q)[0])

## 6. Add your own documents
Drop `.txt`/`.md`/`.pdf` files into `data/`, then rebuild:

In [ ]:
rag.build_index(load_chunks('data'))
print(rag.answer('Summarize my documents in one sentence.')[0])

## 7. Launch the web UI
Run `python app.py` from a terminal.